# Il grafo rispetto ai singoli dataset

Task 5b della traccia: *illustrare come il knowledge graph facilita il recupero delle informazioni rispetto ai singoli dataset*.

Idea: si pongono al grafo alcune domande e, per ogni colonna della risposta, si indica **da quale fonte viene** il dato (dataset 1, 2, 3 e 4). Se una risposta usa più fonti, nessun dataset da solo può darla: senza il grafo bisognerebbe collegare i file a mano, per nome, con gli errori che l'entity resolution del notebook 01 ha già risolto una volta per tutte.

Le domande coprono i casi d'uso del progetto: scheda di un giocatore, squadre di un torneo, analisi tra fonti (scarpe e gol, club e gol con immagini), capitani, immagini scelte con CLIP. L'ultima usa una sola fonte ed è il caso di controllo: lì il grafo non serve a integrare, ma solo ad attraversare quattro tabelle con un percorso.

Il notebook legge soltanto; la tabella finale va in `data/valutazione/risultati/grafo_vs_fonti.csv`.

In [1]:
# [Cella 1] Setup: connessione a Neo4j in sola lettura e percorsi
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from wc_retrieval.grafo import Grafo

OUT = ROOT / "data" / "processed"                         # tabelle del preprocessing (abbinamenti dell'entity resolution)
RIS = ROOT / "data" / "valutazione" / "risultati"
RIS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

# Connessione a Neo4j in sola lettura
g = Grafo(getpass("Password Neo4j: "))

## Le domande
Per ogni domanda: la query Cypher, la fonte di ogni colonna della risposta (`fonti`) e come si dovrebbero collegare i file senza il grafo (`senza_grafo`). La cella controlla che le colonne della risposta siano quelle dichiarate.

In [2]:
# [Cella 2] Domande al grafo e fonte di ogni colonna della risposta
# Sigle delle quattro fonti
FONTI = {"1": "dataset 1", "2": "dataset 2", "3": "dataset 3", "4": "dataset 4"}
# Ogni domanda: query Cypher, fonte di ogni colonna della risposta, come si farebbe senza grafo
DOMANDE = [
 dict(id="G1", domanda="Scheda di Lionel Messi: Mondiali, gol, premi; club, scarpe e assist 2022; capitano; immagine",
      cypher="""
        MATCH (p:Player {label: 'Lionel Messi'})
        WITH p, COUNT { (p)<-[:player]-(:SquadMembership) } AS mondiali,
                COUNT { (p)<-[:player]-(:Goal {isOwnGoal: false}) } AS gol,
                COUNT { (p)<-[:player]-(:SquadMembership)-[:wonAward]->(:Award) } AS premi
        MATCH (p)<-[:player]-(s:SquadMembership)-[:participation]->(:Participation)-[:tournament]->(:Tournament {year: 2022})
        OPTIONAL MATCH (s)-[:club]->(c:Club)
        OPTIONAL MATCH (s)-[:bootBrand]->(b:Brand)
        OPTIONAL MATCH (p)<-[:depicts]-(i:Image {hasDuplicateContent: false})
        WITH p, mondiali, gol, premi, s, c, b, i ORDER BY i.labelConfidence DESC
        RETURN p.label AS giocatore, mondiali, gol, premi, c.label AS club_2022, b.label AS scarpe_2022,
               s.assists AS assist_2022, s.isCaptain AS capitano_2022, collect(i.filePath)[0] AS immagine""",
      fonti={"giocatore": "1", "mondiali": "1", "gol": "1", "premi": "1", "club_2022": "3", "scarpe_2022": "3",
             "assist_2022": "3", "capitano_2022": "4", "immagine": "4"},
      senza_grafo="dataset 1 e dataset 3 per nome del giocatore; dataset 1 e cartelle delle immagini (dataset 4) per nome (nomi corrotti, capitano nel nome della cartella)"),
 dict(id="G2", domanda="Le 32 squadre del 2022: posizione finale, fase raggiunta, ranking FIFA, allenatore, sponsor tecnico",
      cypher="""
        MATCH (pa:Participation)-[:tournament]->(:Tournament {year: 2022})
        MATCH (pa)-[:team]->(:NationalTeam)-[:representsFederation]->(f:Federation)
        OPTIONAL MATCH (pa)-[:hasManager]->(m:Manager)
        OPTIONAL MATCH (pa)-[:kitSponsor]->(k:Brand)
        RETURN f.label AS squadra, pa.finalRank AS posizione_finale, pa.performance AS fase_raggiunta,
               pa.fifaRanking AS ranking_fifa, collect(DISTINCT m.label) AS allenatore, k.label AS sponsor_tecnico
        ORDER BY posizione_finale""",
      fonti={"squadra": "1", "posizione_finale": "2", "fase_raggiunta": "1", "ranking_fifa": "3",
             "allenatore": "1", "sponsor_tecnico": "3"},
      senza_grafo="dataset 2 e dataset 1 per nome della squadra (alias come USA e China PR); dataset 3 per nome della nazionale"),
 dict(id="G3", domanda="Gol del Mondiale 2022 per marca di scarpe",
      cypher="""
        MATCH (g:Goal {isOwnGoal: false})-[:participation]->(pa:Participation)-[:tournament]->(:Tournament {year: 2022})
        MATCH (g)-[:player]->(p:Player)<-[:player]-(s:SquadMembership)-[:participation]->(pa)
        MATCH (s)-[:bootBrand]->(b:Brand)
        RETURN b.label AS marca, count(g) AS gol, count(DISTINCT p) AS marcatori
        ORDER BY gol DESC""",
      fonti={"marca": "3", "gol": "1", "marcatori": "1"},
      senza_grafo="gol del dataset 1 e scarpe del dataset 3 collegati per nome del giocatore"),
 dict(id="G4", domanda="Club con più gol nella fase a eliminazione 2022, con i marcatori e le loro immagini",
      cypher="""
        MATCH (g:Goal {isOwnGoal: false})-[:inMatch]->(:Match)-[:inStage]->(:Stage {isKnockoutStage: true})
              -[:stageOf]->(:Tournament {year: 2022})
        MATCH (g)-[:participation]->(pa:Participation)<-[:participation]-(s:SquadMembership)-[:player]->(p:Player)<-[:player]-(g)
        MATCH (s)-[:club]->(c:Club)
        WITH c, count(g) AS gol, collect(DISTINCT p) AS giocatori
        UNWIND giocatori AS x
        WITH c, gol, x, COUNT { (x)<-[:depicts]-(:Image) } AS n
        RETURN c.label AS club, gol, collect(x.label) AS marcatori, sum(n) AS immagini_dei_marcatori
        ORDER BY gol DESC, club LIMIT 10""",
      fonti={"club": "3", "gol": "1", "marcatori": "1", "immagini_dei_marcatori": "4"},
      senza_grafo="gol e fasi del dataset 1, club del dataset 3 (nomi da unificare: 316 grafie per 291 club) e cartelle delle immagini, tutti per nome"),
 dict(id="G5", domanda="Capitani del 2022 e gol segnati in tutti i loro Mondiali",
      cypher="""
        MATCH (s:SquadMembership {isCaptain: true})-[:player]->(p:Player)
        MATCH (s)-[:participation]->(:Participation)-[:team]->(:NationalTeam)-[:representsFederation]->(f:Federation)
        RETURN f.label AS squadra, p.label AS capitano,
               COUNT { (p)<-[:player]-(:SquadMembership) } AS mondiali,
               COUNT { (p)<-[:player]-(:Goal {isOwnGoal: false}) } AS gol_ai_mondiali
        ORDER BY gol_ai_mondiali DESC, mondiali DESC, capitano""",
      fonti={"squadra": "1", "capitano": "4", "mondiali": "1", "gol_ai_mondiali": "1"},
      senza_grafo="il capitano è solo nel nome della cartella delle immagini ('(captain)'); i gol di tutta la carriera sono nel dataset 1"),
 dict(id="G6", domanda="Portieri del Brasile 2022: maglia, percentuale di parate e immagine migliore secondo CLIP",
      cypher="""
        MATCH (:Federation {label: 'Brazil'})<-[:representsFederation]-(:NationalTeam)<-[:team]-(pa:Participation)
              -[:tournament]->(:Tournament {year: 2022})
        MATCH (pa)<-[:participation]-(s:SquadMembership)-[:position]->(:Position {notation: 'GK'})
        MATCH (s)-[:player]->(p:Player)<-[:depicts]-(i:Image {hasDuplicateContent: false})
        WITH p, s, i ORDER BY i.labelConfidence DESC
        WITH p, s, collect(i)[0] AS migliore
        RETURN p.label AS portiere, s.shirtNumber AS maglia, s.savePercentage AS parate_2022,
               migliore.filePath AS immagine, round(migliore.labelConfidence, 3) AS confidenza
        ORDER BY maglia""",
      fonti={"portiere": "1", "maglia": "1", "parate_2022": "3", "immagine": "4", "confidenza": "4"},
      senza_grafo="ruolo del dataset 1, statistiche del dataset 3 e immagini per nome; la confidenza esiste solo dopo il calcolo con CLIP (notebook 04)"),
 dict(id="G7", domanda="Titoli mondiali per confederazione e genere (caso di controllo: una sola fonte)",
      cypher="""
        MATCH (pa:Participation {finalPosition: 1})-[:tournament]->(t:Tournament)
        MATCH (pa)-[:team]->(:NationalTeam)-[:representsFederation]->(:Federation)-[:memberOf]->(c:Confederation)
        RETURN c.confederationCode AS confederazione, t.gender AS genere, count(*) AS titoli
        ORDER BY genere, titoli DESC""",
      fonti={"confederazione": "1", "genere": "1", "titoli": "1"},
      senza_grafo="solo il dataset 1, con quattro join SQL (classifica, squadre, federazioni, confederazioni)"),
]

risposte = {}
# Esegue le query e controlla che le colonne siano quelle dichiarate
for d in DOMANDE:
    r = g.leggi(d["cypher"])
    assert list(r.columns) == list(d["fonti"]), f"{d['id']}: colonne {list(r.columns)} invece di {list(d['fonti'])}"
    risposte[d["id"]] = r
    print(f"\n{d['id']} — {d['domanda']} ({len(r)} righe)")
    print(r.head(8).to_string(index=False))


G1 — Scheda di Lionel Messi: Mondiali, gol, premi; club, scarpe e assist 2022; capitano; immagine (1 righe)
   giocatore  mondiali  gol  premi club_2022 scarpe_2022  assist_2022  capitano_2022                                                                             immagine
Lionel Messi         5   13      3       PSG      Adidas            3           True Group C/Argentina Players/Images_Lionel Messi (captain)/Lionel Messi (captain)17.jpg

G2 — Le 32 squadre del 2022: posizione finale, fase raggiunta, ranking FIFA, allenatore, sponsor tecnico (32 righe)
    squadra  posizione_finale    fase_raggiunta  ranking_fifa         allenatore sponsor_tecnico
  Argentina                 1             final             2   [Lionel Scaloni]          Adidas
     France                 2             final             3 [Didier Deschamps]            Nike
    Croatia                 3 third-place match             7     [Zlatko Dalić]            Nike
    Morocco                 4 third-place matc

## Quale dataset da solo potrebbe rispondere?
Per ogni domanda, la quota delle colonne della risposta che ogni fonte può fornire. Un dataset singolo basta solo se fornisce tutte le colonne.

In [3]:
# [Cella 3] Matrice domanda × dataset e salvataggio
righe = []
for d in DOMANDE:
    fonti = pd.Series(d["fonti"])
    riga = {"id": d["id"], "domanda": d["domanda"], "colonne": len(fonti)}
    for k, nome in FONTI.items():
        riga[nome] = f"{(fonti == k).sum()}/{len(fonti)}"
    # Fonti che servono per rispondere: se è una sola, basta un dataset
    necessarie = [k for k in FONTI if k in set(fonti)]
    riga["fonti necessarie"] = " + ".join(FONTI[k] for k in necessarie)
    riga["un dataset basta"] = "sì" if len(necessarie) == 1 else "no"
    riga["senza grafo"] = d["senza_grafo"]
    righe.append(riga)
matrice = pd.DataFrame(righe)
matrice.to_csv(RIS / "grafo_vs_fonti.csv", index=False, encoding="utf-8")
print(matrice.drop(columns=["domanda", "senza grafo"]).to_string(index=False))
print(f"\ndomande a cui risponde un dataset da solo: {(matrice['un dataset basta'] == 'sì').sum()} su {len(matrice)}")
print("salvato:", RIS / "grafo_vs_fonti.csv")

id  colonne dataset 1 dataset 2 dataset 3 dataset 4                  fonti necessarie un dataset basta
G1        9       4/9       0/9       3/9       2/9 dataset 1 + dataset 3 + dataset 4               no
G2        6       3/6       1/6       2/6       0/6 dataset 1 + dataset 2 + dataset 3               no
G3        3       2/3       0/3       1/3       0/3             dataset 1 + dataset 3               no
G4        4       2/4       0/4       1/4       1/4 dataset 1 + dataset 3 + dataset 4               no
G5        4       3/4       0/4       0/4       1/4             dataset 1 + dataset 4               no
G6        5       2/5       0/5       1/5       2/5 dataset 1 + dataset 3 + dataset 4               no
G7        3       3/3       0/3       0/3       0/3                         dataset 1               sì

domande a cui risponde un dataset da solo: 1 su 7
salvato: <progetto>\data\valutazione\risultati\grafo_vs_fonti.csv


## Il costo di collegare le fonti senza il grafo
Senza grafo, i file si collegano per nome. Gli abbinamenti del notebook 01 dicono quanti giocatori si perderebbero con il solo nome esatto, cioè quanti sono stati recuperati con la somiglianza dei nomi, la data di nascita e la maglia, o a mano. L'ultima query conta i convocati 2022 che nel grafo hanno i dati di tutte le fonti.

In [4]:
# [Cella 4] Nome esatto contro entity resolution; convocati 2022 con i dati di tutte le fonti
# Quanti giocatori si collegherebbero con il solo nome esatto
for nome, file in [("dataset 3 (righe)", "match_dataset_3.csv"), ("immagini (cartelle)", "match_images.csv")]:
    m = pd.read_csv(OUT / file)
    esatti = (m["metodo"] == "1_nome_esatto").sum()
    recuperati = m["player_id"].notna().sum() - esatti
    print(f"{nome}: {len(m)} | nome esatto {esatti} ({esatti / len(m):.1%}) | recuperati dall'entity resolution "
          f"{recuperati} | senza giocatore {m['player_id'].isna().sum()}")

# Convocati 2022 che nel grafo hanno i dati di tutte le fonti
print(g.leggi("""
    MATCH (s:SquadMembership)-[:participation]->(:Participation)-[:tournament]->(:Tournament {year: 2022})
    MATCH (s)-[:player]->(p:Player)
    WITH s, p,
         EXISTS { (s)-[:club]->() } OR EXISTS { (s)-[:bootBrand]->() } OR s.reportedAppearances IS NOT NULL AS dataset_3,
         EXISTS { (p)<-[:depicts]-() } AS immagini
    RETURN count(*) AS convocati_2022,
           sum(CASE WHEN dataset_3 THEN 1 ELSE 0 END) AS con_dati_dataset_3,
           sum(CASE WHEN immagini THEN 1 ELSE 0 END) AS con_immagini,
           sum(CASE WHEN dataset_3 AND immagini THEN 1 ELSE 0 END) AS con_tutte_le_fonti""").to_string(index=False))

dataset 3 (righe): 814 | nome esatto 766 (94.1%) | recuperati dall'entity resolution 48 | senza giocatore 0
immagini (cartelle): 831 | nome esatto 821 (98.8%) | recuperati dall'entity resolution 9 | senza giocatore 1
 convocati_2022  con_dati_dataset_3  con_immagini  con_tutte_le_fonti
            831                 814           829                 812
